# Test — 4 synthetic windows + 5 simulated overlaps
Explicit technology="test", assay-specific calibration: DLPFC 138 and MERFISH 100 pixels per 100 μm. Uses existing feature costs and the experimental router. Saves matrices, four metrics, route decisions and comparisons with historical reference matrices. Historical equality is reported, not assumed.

In [ ]:
"""Exercise public interface for all nine cases and compare saved reference matrices."""
import sys,json,time,subprocess
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, '/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/1_simulated_data_exp')
import unified_partial_overlap_benchmark as base
import hashlib, inspect
from raftup import _cutoff_selection as current_cs
source_root = Path("/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src/raftup")
assert Path(current_cs.__file__).resolve() == (source_root / "_cutoff_selection.py").resolve(), "Restart kernel and import local repo"
assert inspect.signature(current_cs.run_automated_alignment).parameters["technology"].default == "dlpfc"
assert "pixels_per_100um" in inspect.signature(current_cs.run_automated_alignment).parameters, "Restart kernel: outdated main function"
SOURCE_FINGERPRINTS = {str(p.relative_to(source_root)): hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in sorted(source_root.rglob("*")) if p.is_file() and p.suffix in {".py", ".pkl"}}

OUT=Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/test');OUT.mkdir(exist_ok=True)


In [ ]:
def run_case(name):
    a,b,cost,tech=base.load_condition(name);folder=OUT/name;folder.mkdir(parents=True,exist_ok=True)
    (folder/'source_fingerprints.json').write_text(json.dumps(SOURCE_FINGERPRINTS,indent=2))
    start=time.perf_counter();p,d=base.cs.run_automated_alignment(a,b,cost,technology='test',data_technology=tech,pixels_per_100um=138 if tech=='dlpfc' else 100,output_dir=folder)
    route='synthetic_window' if name in base.WINDOWS else 'simulated_localized'
    assert d['selected_workflow']==route,(name,d['selected_workflow'])
    prior=(base.REPO/'separate_overlap_classifiers'/name/'alignment_matrix.npy') if name in base.WINDOWS else (base.REPO/'unified_simulated_localization'/name/'alignment_matrix.npy')
    old=np.load(prior);assert p.shape==old.shape and np.isfinite(p).all() and (p>=0).all()
    identical=np.array_equal(p,old);support_equal=np.array_equal(p>0,old>0)
    scores=base.evaluate(name,a,b,p,{})
    result={'condition':name,'route':route,'actual_technology':tech,'inferred_technology':base.cs._infer_technology(a,b),'identical_matrix':identical,'same_support':support_equal,'max_abs_difference':float(np.abs(p-old).max()),'reference':str(prior),'seconds':time.perf_counter()-start,**{k:scores[k] for k in ['LAA','GPR','GPR_radius_coordinate_units','coverage_A','coverage_B']}}
    assert np.array_equal(np.load(folder/'alignment_matrix.npy'),p)
    assert json.loads((folder/'cutoff_decision.json').read_text())['selected_workflow']==route
    (folder/'verification.json').write_text(json.dumps(result,indent=2));print(json.dumps(result),flush=True)
    return result

In [ ]:
SELECTED_CASES = list(base.CONDITIONS)
results = []
for name in SELECTED_CASES:
    results.append(run_case(name))
    pd.DataFrame(results).to_csv(OUT / "verification.csv", index=False)
display(pd.DataFrame(results))